# CNU Campus ChatBot — 분류기 평가 진입점 (Task 1)

**PDF p10, p11 명세**:
- 입력: `/data/test_cls.json` (`[{"question":...}]`)
- 출력: `/outputs/cls_output.json` (`[{"question":..., "label": int 0~4}]`)
- 라벨: 0=졸업요건 / 1=공지 / 2=학사일정 / 3=식단 / 4=셔틀

셀 순서대로 실행 (`Run All`).

In [ ]:
# [Cell 0] 셋업 — Drive 마운트 + zip 탐지/해제 + 의존성
import os, sys, glob, zipfile, shutil, subprocess
from pathlib import Path

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except ImportError:
    pass  # 로컬 환경

import torch
assert torch.cuda.is_available(), '❌ T4 GPU 선택 필요'
print(f'✅ {torch.cuda.get_device_name(0)} '
      f'({torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB) / '
      f'torch={torch.__version__}')

# PROJECT_ROOT 자동 탐지 — (a) 노트북이 src/ 안에서 실행되는 케이스,
#                        (b) Drive 의 zip 을 풀어야 하는 케이스 둘 다 대응
_here = Path(os.getcwd()).resolve()
if (_here.parent / 'chatbot.sh').exists():       # nbconvert: cwd = src/
    PROJECT_ROOT = _here.parent
elif (_here / 'chatbot.sh').exists():            # cwd = project root
    PROJECT_ROOT = _here
else:                                            # Drive zip 풀기
    zips = sorted(glob.glob('/content/drive/MyDrive/Termproject_*final*.zip')
                  or glob.glob('/content/drive/MyDrive/Termproject_*.zip'))
    assert zips, '❌ Drive 에 Termproject_*.zip 없음'
    extract = Path('/content/workspace')
    if extract.exists(): shutil.rmtree(extract)
    with zipfile.ZipFile(zips[-1]) as z: z.extractall(extract)
    PROJECT_ROOT = next((d for d in extract.iterdir() if d.is_dir()), extract)
    print(f'📦 {zips[-1]}')
print(f'📁 PROJECT_ROOT = {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/content/hf_cache/hub'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# 충돌 패키지 선제 제거 (현재 미설치라 보통 no-op)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','-q',
                'torchao','torchcodec'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 의존성 체크 — 모두 import 가능하면 skip
check = subprocess.run(
    [sys.executable, '-c',
     'import fastapi, uvicorn, transformers, faiss, bitsandbytes, '
     'sentence_transformers, peft, accelerate, lxml'],
    capture_output=True)
if check.returncode != 0:
    print('[deps] pip install (~1~2분)')
    subprocess.run(
        [sys.executable,'-m','pip','install','-q','--no-cache-dir',
         '--upgrade-strategy','only-if-needed','--prefer-binary',
         '-r', str(PROJECT_ROOT / 'requirements.txt')])
else:
    print('[deps] 사전설치 충족 — pip 건너뜀')

# sys.modules 정리 (bitsandbytes 는 torch.library op 중복 등록 회피 위해 제외)
for m in list(sys.modules):
    if m.startswith(('torchvision','torchaudio','torchao','torchcodec',
                     'transformers','sentence_transformers','timm')):
        del sys.modules[m]

import transformers, bitsandbytes, sentence_transformers, faiss, fastapi
print(f'\n🎉 OK — torch={torch.__version__} / '
      f'transformers={transformers.__version__} / bnb={bitsandbytes.__version__}')


In [ ]:
# [Cell 1] Task 1 — 분류기 실행 → outputs/cls_output.json
import sys, time, json
from pathlib import Path

# chat_model 재import 보장 (Cell 0 의 sys.modules 정리 후)
for m in list(sys.modules):
    if m.startswith(('chat_model','cnubot')): del sys.modules[m]

from chat_model import classify_batch

IN  = PROJECT_ROOT / 'data' / 'test_cls.json'
OUT = PROJECT_ROOT / 'outputs' / 'cls_output.json'
assert IN.exists(), f'평가 입력 없음: {IN}'

print(f'[infer] {IN}')
t0 = time.time()
classify_batch(str(IN), str(OUT))
print(f'[done] {time.time()-t0:.0f}s — {OUT}')


In [ ]:
# [Cell 2] 최소 합격선 검증 (PDF p11)
import json
res = json.load(open(OUT, encoding='utf-8'))
n_in = len(json.load(open(IN, encoding='utf-8')))
ok = True
def check(c, m):
    global ok
    print(('✅' if c else '❌'), m); ok = ok and c

check(OUT.exists(),                                           f'파일 생성 — {OUT.name}')
check(len(res) == n_in,                                       f'건수 일치 — {len(res)} / {n_in}')
check(all(set(r.keys()) >= {"question","label"} for r in res),'키 = {question, label}')
check(all(isinstance(r['label'], int) for r in res),          'label 타입 = int')
check(all(0 <= r['label'] <= 4 for r in res),                 'label 범위 = 0~4')
dist = {i: sum(1 for r in res if r['label']==i) for i in range(5)}
check(len([v for v in dist.values() if v>0]) >= 3,            f'라벨 다양성 ({dist})')
print('\n' + ('🎉 Task 1 PASS' if ok else '🛑 Task 1 FAIL'))
for r in res[:5]: print(f"  [{r['label']}] {r['question']}")
